# Coach DNA Profile Checks

This notebook is the main findings notebook for the upgraded Coach DNA model.

The first pass of the project showed that down, distance, score, and clock matter.  
The upgraded model adds **field position** so coaching behavior is measured at the **situation + field zone** level.

This notebook answers the bigger question:

**How can real NFL coaching behavior be translated into more authentic CPU-controlled coaching logic?**

## What this notebook does
- validates the upgraded scoring outputs
- identifies the strongest team-level coaching DNA signals
- shows which field-zone contexts create the biggest separation across teams
- highlights team-context combinations that could inform CPU tuning
- turns technical outputs into findings for README, GitHub, and LinkedIn

In [84]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 140)

# Robust project root detection
candidate_paths = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = None

for path in candidate_paths:
    if (path / "python").exists() and (path / "data").exists():
        PROJECT_ROOT = path
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root from notebook.")

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)
print("OUTPUT_TABLES_DIR:", OUTPUT_TABLES_DIR)

PROJECT_ROOT: /Users/Tip/Desktop/ea-coach-dna-calibration
PROCESSED_DATA_DIR: /Users/Tip/Desktop/ea-coach-dna-calibration/data/processed
OUTPUT_TABLES_DIR: /Users/Tip/Desktop/ea-coach-dna-calibration/outputs/tables


In [85]:
ranked_team_summary = pd.read_csv(
    OUTPUT_TABLES_DIR / "coach_dna_ranked_team_summary_2025.csv"
)

ranked_situation_scores = pd.read_csv(
    OUTPUT_TABLES_DIR / "coach_dna_ranked_situation_scores_2025.csv"
)

top_signal_situations = pd.read_csv(
    OUTPUT_TABLES_DIR / "coach_dna_top_signal_situations_by_team_2025.csv"
)

presentation_summary = pd.read_csv(
    OUTPUT_TABLES_DIR / "coach_dna_team_summary_presentation_2025.csv"
)

situation_strength_summary = pd.read_csv(
    OUTPUT_TABLES_DIR / "coach_dna_situation_strength_summary_2025.csv"
)

team_baseline_features = pd.read_csv(
    PROCESSED_DATA_DIR / "team_baseline_features_2025_vs_2023_2025.csv"
)

coach_dna_situation_scores = pd.read_csv(
    PROCESSED_DATA_DIR / "coach_dna_situation_scores_2025.csv"
)

coach_dna_team_summary = pd.read_csv(
    PROCESSED_DATA_DIR / "coach_dna_team_summary_2025.csv"
)

print("ranked_team_summary:", ranked_team_summary.shape)
print("ranked_situation_scores:", ranked_situation_scores.shape)
print("top_signal_situations:", top_signal_situations.shape)
print("presentation_summary:", presentation_summary.shape)
print("situation_strength_summary:", situation_strength_summary.shape)
print("team_baseline_features:", team_baseline_features.shape)
print("coach_dna_situation_scores:", coach_dna_situation_scores.shape)
print("coach_dna_team_summary:", coach_dna_team_summary.shape)

ranked_team_summary: (32, 18)
ranked_situation_scores: (2458, 27)
top_signal_situations: (96, 23)
presentation_summary: (32, 18)
situation_strength_summary: (78, 13)
team_baseline_features: (2458, 65)
coach_dna_situation_scores: (2458, 65)
coach_dna_team_summary: (32, 17)


## 1. Basic QA Checks
These checks confirm the exported tables look structurally right before interpreting the results.

In [86]:
qa_summary = {
    "team_summary_rows": len(ranked_team_summary),
    "unique_teams_in_team_summary": ranked_team_summary["team"].nunique(),
    "situation_score_rows": len(ranked_situation_scores),
    "unique_teams_in_situation_scores": ranked_situation_scores["team"].nunique(),
    "unique_situations_in_situation_scores": ranked_situation_scores["situation_name"].nunique(),
    "unique_field_zones_in_situation_scores": ranked_situation_scores["field_zone"].nunique(),
    "unique_contexts_in_situation_scores": ranked_situation_scores["situation_field_zone_context"].nunique(),
    "top_signal_rows": len(top_signal_situations),
    "unique_teams_in_top_signals": top_signal_situations["team"].nunique(),
}

pd.Series(qa_summary)

team_summary_rows                           32
unique_teams_in_team_summary                32
situation_score_rows                      2458
unique_teams_in_situation_scores            32
unique_situations_in_situation_scores       22
unique_field_zones_in_situation_scores       4
unique_contexts_in_situation_scores         78
top_signal_rows                             96
unique_teams_in_top_signals                 32
dtype: int64

In [87]:
ranked_situation_scores.groupby(["situation_name", "field_zone"]).size().sort_values().head(20)

situation_name            field_zone   
fourth_down               backed_up         9
two_minute_game           backed_up        28
leading_two_plus_scores   backed_up        29
short_yardage             backed_up        31
two_minute_half           backed_up        31
two_minute_game           red_zone         31
leading_two_plus_scores   fringe           31
                          own_territory    31
                          red_zone         31
fourth_down               own_territory    31
trailing_two_plus_scores  backed_up        31
tied_early_down           red_zone         32
                          own_territory    32
                          fringe           32
tied                      red_zone         32
trailing                  backed_up        32
tied                      own_territory    32
                          fringe           32
                          backed_up        32
third_down                red_zone         32
dtype: int64

In [88]:
ranked_situation_scores.groupby("team").size().sort_values(ascending=False).head(10)

team
ARI    78
CIN    78
DAL    78
MIA    78
KC     77
TEN    77
TB     77
SF     77
PHI    77
NYG    77
dtype: int64

## 2. Team-Level Results

Start with the overall rankings to see which teams stand out most strongly in the upgraded field-zone-aware Coach DNA model.

In [89]:
ranked_team_summary.head(10)

,rank,team,overall_coach_dna_score,score_tier,scored_situations,tendency_signal_score_avg,efficiency_signal_score_avg,explosiveness_signal_score_avg,stability_signal_score_avg,sample_reliability_score_avg,top_signal_context,top_signal_situation,top_signal_field_zone,top_signal_score,lowest_signal_context,lowest_signal_situation,lowest_signal_field_zone,lowest_signal_score
0,1,LA,56.2940,strong_signal,45,62.4652,72.6318,65.3343,62.2810,73.6662,neutral_early_down | fringe,neutral_early_down,fringe,80.6250,two_minute_game | own_territory,two_minute_game,own_territory,17.2246
1,2,BUF,53.1461,solid_signal,45,63.8664,63.7499,62.2197,47.1380,72.1871,leading_early_down | own_territory,leading_early_down,own_territory,83.5156,tied_early_down | backed_up,tied_early_down,backed_up,16.5801
2,3,WAS,51.9778,solid_signal,45,73.1083,52.7241,59.8365,47.8622,69.8413,neutral_early_down | own_territory,neutral_early_down,own_territory,83.3984,two_minute_game | red_zone,two_minute_game,red_zone,23.4113
3,4,CIN,50.8784,solid_signal,46,64.2256,60.0367,47.1958,53.3315,73.8247,trailing_one_score | fringe,trailing_one_score,fringe,71.6406,two_minute_game | backed_up,two_minute_game,backed_up,17.9487
4,5,BAL,49.1322,solid_signal,45,61.2633,54.7651,63.6064,35.2844,71.8062,neutral_early_down | fringe,neutral_early_down,fringe,77.4219,trailing_early_down | backed_up,trailing_early_down,backed_up,11.8242
5,6,NE,49.0881,solid_signal,45,49.5704,68.2048,63.1267,44.2766,75.1637,leading_one_score | fringe,leading_one_score,fringe,66.8164,fourth_down | own_territory,fourth_down,own_territory,14.7419
6,7,CHI,48.8438,solid_signal,45,50.1781,59.5375,61.3148,65.6443,74.7450,trailing_one_score | own_territory,trailing_one_score,own_territory,68.3203,two_minute_half | backed_up,two_minute_half,backed_up,12.8468
7,8,SF,48.3072,solid_signal,45,55.0335,60.0075,47.3050,57.9523,74.2230,neutral_early_down | own_territory,neutral_early_down,own_territory,75.7227,fourth_down | own_territory,fourth_down,own_territory,11.9093
8,9,SEA,47.6358,solid_signal,44,58.7242,56.5951,53.5932,47.8587,70.4528,third_down | own_territory,third_down,own_territory,70.9242,two_minute_half | red_zone,two_minute_half,red_zone,14.1875
9,10,DET,46.0869,moderate_signal,43,49.1242,58.7184,56.7840,55.7656,73.5606,leading_early_down | own_territory,leading_early_down,own_territory,68.3594,fourth_down | own_territory,fourth_down,own_territory,18.4718


### How to read this table

This table shows the teams with the strongest overall Coach DNA signal in the upgraded model.

#### What the key numbers mean

- `overall_coach_dna_score`  
  Higher means the team shows a stronger and more credible offensive identity relative to the league baseline.

- `tendency_signal_score_avg`  
  Higher means the team behaves more differently from baseline across weighted contexts.

- `efficiency_signal_score_avg`  
  Higher means the team’s distinct behavior tends to work better than baseline.

- `top_signal_context`  
  This shows the exact **situation + field-zone context** where the team’s offensive coaching DNA appears most clearly.

#### Why this matters for CPU-controlled coaching decisions

A high-ranking team in this table is a team where the CPU should probably not be tuned as a league-average generalist.

Instead, it suggests the team has stronger evidence for:
- context-specific run/pass behavior
- field-position-aware tempo shifts
- distinct formation preferences
- more realistic offensive identity from one part of the field to another

### Practical takeaway

This table shows which teams have the strongest evidence for **custom CPU coach tuning**, not just custom tuning by situation, but tuning by **situation + field position**.

In [90]:
ranked_team_summary.tail(10)

,rank,team,overall_coach_dna_score,score_tier,scored_situations,tendency_signal_score_avg,efficiency_signal_score_avg,explosiveness_signal_score_avg,stability_signal_score_avg,sample_reliability_score_avg,top_signal_context,top_signal_situation,top_signal_field_zone,top_signal_score,lowest_signal_context,lowest_signal_situation,lowest_signal_field_zone,lowest_signal_score
22,23,CAR,39.8156,developing_signal,45,50.9468,41.6961,42.6072,51.5844,72.0006,neutral_early_down | own_territory,neutral_early_down,own_territory,60.8594,leading_early_down | red_zone,leading_early_down,red_zone,14.9004
23,24,MIN,39.6160,developing_signal,45,50.7574,43.6704,53.7345,31.8954,69.9167,neutral_early_down | fringe,neutral_early_down,fringe,58.3984,fourth_down | own_territory,fourth_down,own_territory,13.7238
24,25,HOU,39.5336,developing_signal,45,45.3997,40.5529,43.2687,61.9362,73.4961,trailing_early_down | own_territory,trailing_early_down,own_territory,66.4844,goal_line | red_zone,goal_line,red_zone,16.0078
25,26,NO,39.4741,developing_signal,45,52.8836,39.3238,41.5848,44.9186,71.1295,red_zone | red_zone,red_zone,red_zone,60.1313,goal_line | red_zone,goal_line,red_zone,14.7930
26,27,LAC,38.7050,developing_signal,45,42.9107,47.6455,45.2539,40.1257,74.3923,trailing_early_down | fringe,trailing_early_down,fringe,63.1250,goal_line | red_zone,goal_line,red_zone,16.5703
27,28,PIT,37.9531,developing_signal,44,37.1086,52.7544,43.6141,57.7443,73.0790,leading_early_down | own_territory,leading_early_down,own_territory,58.3984,tied_early_down | backed_up,tied_early_down,backed_up,17.2930
28,29,NYJ,36.3375,developing_signal,46,53.6502,33.8928,35.9596,40.7223,68.3205,tied_early_down | own_territory,tied_early_down,own_territory,60.9223,two_minute_game | own_territory,two_minute_game,own_territory,15.5254
29,30,TEN,35.6780,developing_signal,46,47.4456,34.8692,37.5952,49.2146,70.2733,leading_one_score | own_territory,leading_one_score,own_territory,55.0512,two_minute_half | backed_up,two_minute_half,backed_up,14.7218
30,31,LV,32.9890,developing_signal,46,46.0551,33.1972,38.6016,37.8637,68.6430,trailing_early_down | red_zone,trailing_early_down,red_zone,55.7367,tied_early_down | red_zone,tied_early_down,red_zone,15.1934
31,32,CLE,32.4300,developing_signal,45,40.7636,28.1559,37.6436,47.6155,72.2671,tied_early_down | fringe,tied_early_down,fringe,51.1934,trailing_one_score | backed_up,trailing_one_score,backed_up,11.5996


In [91]:
ranked_team_summary["score_tier"].value_counts()

score_tier
moderate_signal      13
developing_signal    10
solid_signal          8
strong_signal         1
Name: count, dtype: int64

In [92]:
ranked_team_summary[[
    "rank",
    "team",
    "overall_coach_dna_score",
    "score_tier",
    "tendency_signal_score_avg",
    "efficiency_signal_score_avg",
    "explosiveness_signal_score_avg",
    "stability_signal_score_avg",
    "top_signal_context",
    "top_signal_score",
    "lowest_signal_context",
    "lowest_signal_score",
]].head(15)

,rank,team,overall_coach_dna_score,score_tier,tendency_signal_score_avg,efficiency_signal_score_avg,explosiveness_signal_score_avg,stability_signal_score_avg,top_signal_context,top_signal_score,lowest_signal_context,lowest_signal_score
0,1,LA,56.2940,strong_signal,62.4652,72.6318,65.3343,62.2810,neutral_early_down | fringe,80.6250,two_minute_game | own_territory,17.2246
1,2,BUF,53.1461,solid_signal,63.8664,63.7499,62.2197,47.1380,leading_early_down | own_territory,83.5156,tied_early_down | backed_up,16.5801
2,3,WAS,51.9778,solid_signal,73.1083,52.7241,59.8365,47.8622,neutral_early_down | own_territory,83.3984,two_minute_game | red_zone,23.4113
3,4,CIN,50.8784,solid_signal,64.2256,60.0367,47.1958,53.3315,trailing_one_score | fringe,71.6406,two_minute_game | backed_up,17.9487
4,5,BAL,49.1322,solid_signal,61.2633,54.7651,63.6064,35.2844,neutral_early_down | fringe,77.4219,trailing_early_down | backed_up,11.8242
5,6,NE,49.0881,solid_signal,49.5704,68.2048,63.1267,44.2766,leading_one_score | fringe,66.8164,fourth_down | own_territory,14.7419
6,7,CHI,48.8438,solid_signal,50.1781,59.5375,61.3148,65.6443,trailing_one_score | own_territory,68.3203,two_minute_half | backed_up,12.8468
7,8,SF,48.3072,solid_signal,55.0335,60.0075,47.3050,57.9523,neutral_early_down | own_territory,75.7227,fourth_down | own_territory,11.9093
8,9,SEA,47.6358,solid_signal,58.7242,56.5951,53.5932,47.8587,third_down | own_territory,70.9242,two_minute_half | red_zone,14.1875
9,10,DET,46.0869,moderate_signal,49.1242,58.7184,56.7840,55.7656,leading_early_down | own_territory,68.3594,fourth_down | own_territory,18.4718


## 3. Situation + Field-Zone Patterns

These checks show which **exact contexts** tend to generate the strongest signals across the league.

In [93]:
situation_strength_summary.head(20)

,rank,situation_order,situation_name,field_zone_order,field_zone,situation_field_zone_context,avg_adjusted_score,max_adjusted_score,min_adjusted_score,avg_team_play_count,team_count,strong_sample_teams,good_or_better_teams
0,1,1,all_offense,2,own_territory,all_offense | own_territory,53.9844,83.0078,25.7812,455.1,32,32,32
1,2,1,all_offense,3,fringe,all_offense | fringe,53.9844,82.6172,30.0781,331.4,32,32,32
2,3,1,all_offense,4,red_zone,all_offense | red_zone,53.9844,82.0703,17.5000,160.7,32,32,32
3,4,2,early_down,2,own_territory,early_down | own_territory,53.9844,84.9609,24.9805,357.4,32,32,32
4,5,2,early_down,3,fringe,early_down | fringe,53.9844,79.5703,23.7891,248.8,32,32,32
5,6,11,one_score,2,own_territory,one_score | own_territory,53.9844,81.7969,32.1094,305.0,32,32,32
6,7,11,one_score,3,fringe,one_score | fringe,53.9844,83.1250,30.1172,217.5,32,32,32
7,8,12,neutral_early_down,2,own_territory,neutral_early_down | own_territory,53.9844,83.3984,31.8945,230.7,32,32,32
8,9,15,trailing,2,own_territory,trailing | own_territory,53.9844,78.5547,29.6875,225.4,32,32,32
9,10,12,neutral_early_down,3,fringe,neutral_early_down | fringe,53.7940,80.6250,29.1211,155.1,32,31,32


### How to read this table

This table shows which **situation + field-zone contexts** are most useful for detecting coaching identity across teams.

#### What the key numbers mean

- `avg_adjusted_score`  
  Higher means that context tends to separate teams more clearly from one another.

- `avg_team_play_count`  
  Higher means the context has enough volume to trust more confidently.

- `strong_sample_teams` and `good_or_better_teams`  
  These show how many teams have enough sample to make the context useful for modeling.

#### Why this matters for CPU-controlled coaching decisions

Not all context in football is equally valuable for tuning.

The strongest tuning contexts are the ones where:
- teams behave differently
- the signal is stable
- the sample is large enough to trust

Those are the best candidates for:
- playcall weighting adjustments
- run/pass balance tuning
- field-position-aware aggression
- more realistic team personality logic

### Practical takeaway

This table helps identify the exact contexts that matter most when building **team-specific CPU coaching profiles**.

In [94]:
situation_strength_summary[
    [
        "rank",
        "situation_field_zone_context",
        "avg_adjusted_score",
        "max_adjusted_score",
        "min_adjusted_score",
        "avg_team_play_count",
        "good_or_better_teams",
    ]
].head(20)

,rank,situation_field_zone_context,avg_adjusted_score,max_adjusted_score,min_adjusted_score,avg_team_play_count,good_or_better_teams
0,1,all_offense | own_territory,53.9844,83.0078,25.7812,455.1,32
1,2,all_offense | fringe,53.9844,82.6172,30.0781,331.4,32
2,3,all_offense | red_zone,53.9844,82.0703,17.5000,160.7,32
3,4,early_down | own_territory,53.9844,84.9609,24.9805,357.4,32
4,5,early_down | fringe,53.9844,79.5703,23.7891,248.8,32
5,6,one_score | own_territory,53.9844,81.7969,32.1094,305.0,32
6,7,one_score | fringe,53.9844,83.1250,30.1172,217.5,32
7,8,neutral_early_down | own_territory,53.9844,83.3984,31.8945,230.7,32
8,9,trailing | own_territory,53.9844,78.5547,29.6875,225.4,32
9,10,neutral_early_down | fringe,53.7940,80.6250,29.1211,155.1,32


## 4. Top Signal Contexts by Team

This helps identify the **situation + field-zone contexts** where each team looks most distinct relative to baseline.

In [95]:
top_signal_situations.head(20)

,team,team_rank_within_top_signals,situation_name,field_zone,situation_field_zone_context,team_play_count,team_sample_quality,coach_dna_score_adjusted,tendency_signal_score,efficiency_signal_score,explosiveness_signal_score,stability_signal_score,dropback_rate_delta,rush_rate_delta,shotgun_rate_delta,no_huddle_rate_delta,avg_epa_delta,success_rate_delta,explosive_play_rate_delta,tendency_profile_label,formation_profile_label,tempo_profile_label,efficiency_profile_label
0,ARI,1,early_down,red_zone,early_down | red_zone,118,strong,68.710938,78.906250,60.937500,61.458333,37.50000,0.2278,-0.2278,0.1794,0.0465,0.0980,-0.0269,0.0148,more_dropback_heavy_than_baseline,more_shotgun_than_baseline,faster_than_baseline,more_efficient_than_baseline
1,ARI,2,trailing_early_down,red_zone,trailing_early_down | red_zone,73,good,65.967188,85.937500,62.500000,66.145833,50.78125,0.2248,-0.2248,0.1377,0.1018,0.1736,-0.0038,0.0293,more_dropback_heavy_than_baseline,more_shotgun_than_baseline,faster_than_baseline,more_efficient_than_baseline
2,ARI,3,red_zone,red_zone,red_zone | red_zone,91,good,65.615625,89.062500,67.187500,68.750000,17.18750,0.2899,-0.2899,0.2410,0.0653,0.0322,-0.0325,0.0370,more_dropback_heavy_than_baseline,more_shotgun_than_baseline,faster_than_baseline,more_efficient_than_baseline
3,ATL,1,trailing_early_down,own_territory,trailing_early_down | own_territory,137,strong,74.804688,61.718750,78.125000,96.875000,79.68750,0.0514,-0.0514,0.1351,0.0514,0.2329,0.1052,0.1011,more_dropback_heavy_than_baseline,more_shotgun_than_baseline,faster_than_baseline,more_efficient_than_baseline
4,ATL,2,trailing,own_territory,trailing | own_territory,173,strong,73.613281,67.968750,65.234375,91.666667,79.68750,0.0589,-0.0589,0.1190,0.0673,0.0905,0.0731,0.0763,more_dropback_heavy_than_baseline,more_shotgun_than_baseline,faster_than_baseline,more_efficient_than_baseline
5,ATL,3,trailing_one_score,own_territory,trailing_one_score | own_territory,88,good,66.600000,69.531250,77.343750,91.666667,56.25000,0.1072,-0.1072,0.1380,0.0052,0.3592,0.1524,0.1323,more_dropback_heavy_than_baseline,more_shotgun_than_baseline,close_to_baseline_tempo,more_efficient_than_baseline
6,BAL,1,one_score,fringe,one_score | fringe,211,strong,83.125000,76.562500,98.437500,90.625000,54.68750,-0.0954,0.0954,-0.0668,-0.0807,0.2823,0.0756,0.0537,more_run_heavy_than_baseline,less_shotgun_than_baseline,slower_than_baseline,more_efficient_than_baseline
7,BAL,2,all_offense,fringe,all_offense | fringe,282,strong,82.617188,79.687500,97.656250,90.625000,37.50000,-0.0866,0.0866,-0.0708,-0.1020,0.2778,0.0703,0.0510,more_run_heavy_than_baseline,less_shotgun_than_baseline,slower_than_baseline,more_efficient_than_baseline
8,BAL,3,early_down,fringe,early_down | fringe,219,strong,79.570312,74.218750,95.312500,89.583333,39.06250,-0.0855,0.0855,-0.0657,-0.1141,0.2162,0.0582,0.0491,more_run_heavy_than_baseline,less_shotgun_than_baseline,slower_than_baseline,more_efficient_than_baseline
9,BUF,1,leading,own_territory,leading | own_territory,154,strong,87.890625,82.812500,96.875000,90.625000,78.12500,-0.0785,0.0785,-0.2334,-0.0476,0.2520,0.0938,0.0706,more_run_heavy_than_baseline,less_shotgun_than_baseline,slower_than_baseline,more_efficient_than_baseline


### How to read this table

This table shows the **contexts** where each team’s coaching DNA appears most strongly.

#### What the key numbers mean

- `coach_dna_score_adjusted`  
  Higher means this exact context is one of the clearest places where the team’s identity shows up.

- `tendency_signal_score`  
  Higher means the team behaves more differently from the baseline in that context.

- `efficiency_signal_score`  
  Higher means the team is outperforming the baseline in that same context.

- profile labels  
  These describe the form of the difference:
  - more run-heavy
  - more dropback-heavy
  - more shotgun
  - faster tempo
  - more efficient than baseline

#### Why this matters for CPU-controlled coaching decisions

This is one of the most useful outputs in the project for a tuning conversation.

It points directly to where a team should feel different from a default CPU coach, not just by game situation, but by **situation + field position**.

### Practical takeaway

This table shows where each team’s **CPU coaching identity is most worth tuning**.

In [96]:
top_signal_situations["situation_name"].value_counts().head(15)

situation_name
all_offense            13
trailing               10
trailing_early_down     9
neutral_early_down      9
early_down              8
trailing_one_score      8
one_score               8
leading                 7
leading_early_down      5
red_zone                3
tied                    3
leading_one_score       3
tied_early_down         2
short_yardage           2
two_minute_half         2
Name: count, dtype: int64

In [97]:
top_signal_situations["tendency_profile_label"].value_counts()

tendency_profile_label
more_dropback_heavy_than_baseline    39
more_run_heavy_than_baseline         35
close_to_baseline_run_pass_split     22
Name: count, dtype: int64

In [98]:
top_signal_situations["efficiency_profile_label"].value_counts()

efficiency_profile_label
more_efficient_than_baseline    78
less_efficient_than_baseline    10
close_to_baseline_efficiency     8
Name: count, dtype: int64

In [99]:
top_signal_situations[[
    "team",
    "team_rank_within_top_signals",
    "situation_field_zone_context",
    "team_play_count",
    "coach_dna_score_adjusted",
    "tendency_profile_label",
    "efficiency_profile_label",
]].head(20)

,team,team_rank_within_top_signals,situation_field_zone_context,team_play_count,coach_dna_score_adjusted,tendency_profile_label,efficiency_profile_label
0,ARI,1,early_down | red_zone,118,68.710938,more_dropback_heavy_than_baseline,more_efficient_than_baseline
1,ARI,2,trailing_early_down | red_zone,73,65.967188,more_dropback_heavy_than_baseline,more_efficient_than_baseline
2,ARI,3,red_zone | red_zone,91,65.615625,more_dropback_heavy_than_baseline,more_efficient_than_baseline
3,ATL,1,trailing_early_down | own_territory,137,74.804688,more_dropback_heavy_than_baseline,more_efficient_than_baseline
4,ATL,2,trailing | own_territory,173,73.613281,more_dropback_heavy_than_baseline,more_efficient_than_baseline
5,ATL,3,trailing_one_score | own_territory,88,66.600000,more_dropback_heavy_than_baseline,more_efficient_than_baseline
6,BAL,1,one_score | fringe,211,83.125000,more_run_heavy_than_baseline,more_efficient_than_baseline
7,BAL,2,all_offense | fringe,282,82.617188,more_run_heavy_than_baseline,more_efficient_than_baseline
8,BAL,3,early_down | fringe,219,79.570312,more_run_heavy_than_baseline,more_efficient_than_baseline
9,BUF,1,leading | own_territory,154,87.890625,more_run_heavy_than_baseline,more_efficient_than_baseline


## 5. Team Deep Dive
Change the `TEAM_CODE` below to inspect any team in more detail.

In [100]:
TEAM_CODE = "BUF"

In [101]:
team_summary_view = ranked_team_summary.loc[ranked_team_summary["team"] == TEAM_CODE]
team_summary_view

,rank,team,overall_coach_dna_score,score_tier,scored_situations,tendency_signal_score_avg,efficiency_signal_score_avg,explosiveness_signal_score_avg,stability_signal_score_avg,sample_reliability_score_avg,top_signal_context,top_signal_situation,top_signal_field_zone,top_signal_score,lowest_signal_context,lowest_signal_situation,lowest_signal_field_zone,lowest_signal_score
1,2,BUF,53.1461,solid_signal,45,63.8664,63.7499,62.2197,47.138,72.1871,leading_early_down | own_territory,leading_early_down,own_territory,83.5156,tied_early_down | backed_up,tied_early_down,backed_up,16.5801


In [102]:
TEAM_CODE = "BUF"

team_summary_view = ranked_team_summary.loc[ranked_team_summary["team"] == TEAM_CODE]
team_summary_view

,rank,team,overall_coach_dna_score,score_tier,scored_situations,tendency_signal_score_avg,efficiency_signal_score_avg,explosiveness_signal_score_avg,stability_signal_score_avg,sample_reliability_score_avg,top_signal_context,top_signal_situation,top_signal_field_zone,top_signal_score,lowest_signal_context,lowest_signal_situation,lowest_signal_field_zone,lowest_signal_score
1,2,BUF,53.1461,solid_signal,45,63.8664,63.7499,62.2197,47.138,72.1871,leading_early_down | own_territory,leading_early_down,own_territory,83.5156,tied_early_down | backed_up,tied_early_down,backed_up,16.5801


In [103]:
team_situations = (
    ranked_situation_scores.loc[ranked_situation_scores["team"] == TEAM_CODE]
    .sort_values(["situation_order", "field_zone_order"])
)

team_situations[[
    "team",
    "situation_name",
    "field_zone",
    "situation_field_zone_context",
    "team_play_count",
    "coach_dna_score_adjusted",
    "tendency_signal_score",
    "efficiency_signal_score",
    "explosiveness_signal_score",
    "stability_signal_score",
    "dropback_rate_delta",
    "rush_rate_delta",
    "shotgun_rate_delta",
    "no_huddle_rate_delta",
    "avg_epa_delta",
    "success_rate_delta",
    "explosive_play_rate_delta",
    "tendency_profile_label",
    "efficiency_profile_label",
]]

,team,situation_name,field_zone,situation_field_zone_context,team_play_count,coach_dna_score_adjusted,tendency_signal_score,efficiency_signal_score,explosiveness_signal_score,stability_signal_score,dropback_rate_delta,rush_rate_delta,shotgun_rate_delta,no_huddle_rate_delta,avg_epa_delta,success_rate_delta,explosive_play_rate_delta,tendency_profile_label,efficiency_profile_label
787,BUF,all_offense,backed_up,all_offense | backed_up,79,48.160547,71.484375,25.000000,53.125000,31.25000,-0.0463,0.0463,-0.1839,-0.0541,-0.1588,-0.0186,-0.0015,close_to_baseline_run_pass_split,less_efficient_than_baseline
5,BUF,all_offense,own_territory,all_offense | own_territory,441,83.007812,78.906250,90.625000,88.541667,65.62500,-0.0669,0.0669,-0.1827,-0.0474,0.2452,0.0812,0.0508,more_run_heavy_than_baseline,more_efficient_than_baseline
49,BUF,all_offense,fringe,all_offense | fringe,346,74.335938,75.781250,73.437500,81.250000,46.87500,-0.0544,0.0544,-0.1796,-0.0870,0.0949,0.0230,0.0294,more_run_heavy_than_baseline,more_efficient_than_baseline
45,BUF,all_offense,red_zone,all_offense | red_zone,190,75.312500,77.343750,91.406250,43.229167,61.71875,-0.0871,0.0871,-0.2484,0.0529,0.1341,0.0723,-0.0091,more_run_heavy_than_baseline,more_efficient_than_baseline
671,BUF,early_down,backed_up,early_down | backed_up,66,50.094141,62.109375,33.593750,63.541667,57.81250,-0.0387,0.0387,-0.1853,-0.0482,-0.0699,-0.0113,0.0187,close_to_baseline_run_pass_split,less_efficient_than_baseline
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
682,BUF,leading_early_down,red_zone,leading_early_down | red_zone,50,49.865625,82.812500,35.937500,18.750000,23.43750,-0.1376,0.1376,-0.3235,0.0709,-0.0145,-0.0110,-0.0545,more_run_heavy_than_baseline,close_to_baseline_efficiency
879,BUF,trailing_early_down,backed_up,trailing_early_down | backed_up,24,46.388672,65.625000,46.093750,69.270833,76.56250,0.0735,-0.0735,-0.2065,-0.0695,0.1225,-0.0461,0.0705,more_dropback_heavy_than_baseline,more_efficient_than_baseline
353,BUF,trailing_early_down,own_territory,trailing_early_down | own_territory,168,57.324219,53.125000,62.109375,52.083333,50.78125,-0.0281,0.0281,-0.1686,-0.0359,0.1155,0.0816,0.0017,close_to_baseline_run_pass_split,more_efficient_than_baseline
50,BUF,trailing_early_down,fringe,trailing_early_down | fringe,126,74.257812,82.031250,73.437500,60.416667,49.21875,-0.0897,0.0897,-0.1486,-0.1450,0.2160,0.0226,0.0195,more_run_heavy_than_baseline,more_efficient_than_baseline


### How to read this table

This section shows how one team’s behavior differs from the league baseline across all major situations **and field zones**.

This is the best section for translating the model into possible CPU tuning ideas.

#### What the key numbers mean

- positive tendency deltas  
  These show where the team behaves differently from baseline in structure, run-pass balance, or tempo.

- positive `avg_epa_delta` and `success_rate_delta`  
  These suggest the team’s tendency is producing stronger results than baseline, which makes it more credible as a tuning signal.

- positive `explosive_play_rate_delta`  
  This suggests the team creates more chunk-play pressure than baseline in that context.

- `coach_dna_score_adjusted`  
  This summarizes how strong and reliable the overall signal is for that team-context combination.

#### Why this matters for CPU-controlled coaching decisions

This section can directly inform questions like:
- should this team call more run or pass in this field position?
- should this team lean more heavily into shotgun here?
- should this team play faster when trailing in this part of the field?
- should this team feel more conservative or more aggressive than baseline in scoring space?

### Practical takeaway

This table is where the project becomes most useful for tuning discussions because it shows **how one specific team should behave differently from league-average CPU logic by context**.

In [104]:
team_situations.sort_values("coach_dna_score_adjusted", ascending=False).head(10)[[
    "situation_field_zone_context",
    "team_play_count",
    "coach_dna_score_adjusted",
    "dropback_rate_delta",
    "rush_rate_delta",
    "avg_epa_delta",
    "success_rate_delta",
    "explosive_play_rate_delta",
    "tendency_profile_label",
    "efficiency_profile_label",
]]

,situation_field_zone_context,team_play_count,coach_dna_score_adjusted,dropback_rate_delta,rush_rate_delta,avg_epa_delta,success_rate_delta,explosive_play_rate_delta,tendency_profile_label,efficiency_profile_label
0,leading | own_territory,154,87.890625,-0.0785,0.0785,0.2520,0.0938,0.0706,more_run_heavy_than_baseline,more_efficient_than_baseline
2,leading_early_down | own_territory,127,83.515625,-0.0650,0.0650,0.2434,0.1001,0.0527,more_run_heavy_than_baseline,more_efficient_than_baseline
5,all_offense | own_territory,441,83.007812,-0.0669,0.0669,0.2452,0.0812,0.0508,more_run_heavy_than_baseline,more_efficient_than_baseline
8,early_down | own_territory,356,82.285156,-0.0619,0.0619,0.1535,0.0738,0.0322,more_run_heavy_than_baseline,more_efficient_than_baseline
12,one_score | own_territory,273,81.093750,-0.0572,0.0572,0.2531,0.0708,0.0613,more_run_heavy_than_baseline,more_efficient_than_baseline
16,one_score | red_zone,108,80.000000,-0.0762,0.0762,0.1865,0.1140,0.0154,more_run_heavy_than_baseline,more_efficient_than_baseline
17,neutral_early_down | own_territory,211,79.785156,-0.0477,0.0477,0.1511,0.0455,0.0528,close_to_baseline_run_pass_split,more_efficient_than_baseline
37,early_down | fringe,259,76.406250,-0.0683,0.0683,0.1433,0.0269,0.0349,more_run_heavy_than_baseline,more_efficient_than_baseline
38,leading_one_score | own_territory,82,76.092188,-0.0790,0.0790,0.2801,0.1008,0.1134,more_run_heavy_than_baseline,more_efficient_than_baseline
45,all_offense | red_zone,190,75.312500,-0.0871,0.0871,0.1341,0.0723,-0.0091,more_run_heavy_than_baseline,more_efficient_than_baseline


In [105]:
team_situations.sort_values("coach_dna_score_adjusted", ascending=True).head(10)[[
    "situation_field_zone_context",
    "team_play_count",
    "coach_dna_score_adjusted",
    "dropback_rate_delta",
    "rush_rate_delta",
    "avg_epa_delta",
    "success_rate_delta",
    "explosive_play_rate_delta",
    "tendency_profile_label",
    "efficiency_profile_label",
]]

,situation_field_zone_context,team_play_count,coach_dna_score_adjusted,dropback_rate_delta,rush_rate_delta,avg_epa_delta,success_rate_delta,explosive_play_rate_delta,tendency_profile_label,efficiency_profile_label
2411,tied_early_down | backed_up,14,16.580078,0.0145,-0.0145,-0.4524,-0.1069,0.0875,close_to_baseline_run_pass_split,less_efficient_than_baseline
2409,third_down | backed_up,13,16.697266,-0.0182,0.0182,-0.6277,-0.0611,-0.0955,close_to_baseline_run_pass_split,less_efficient_than_baseline
2391,fourth_down | fringe,19,17.537109,0.0315,-0.0315,-0.4798,-0.0699,-0.0053,close_to_baseline_run_pass_split,less_efficient_than_baseline
2374,tied | backed_up,15,18.416016,-0.0161,0.0161,-0.5009,-0.1176,0.0693,close_to_baseline_run_pass_split,less_efficient_than_baseline
2254,two_minute_game | backed_up,1,22.625000,0.3012,-0.3012,0.0330,-0.4710,-0.1776,more_dropback_heavy_than_baseline,more_efficient_than_baseline
2124,two_minute_game | red_zone,10,25.608871,-0.3000,0.3000,-0.3763,-0.0783,-0.0433,more_run_heavy_than_baseline,less_efficient_than_baseline
2108,fourth_down | red_zone,9,26.003906,0.0799,-0.0799,1.2521,0.0998,-0.0380,more_dropback_heavy_than_baseline,more_efficient_than_baseline
2015,trailing_one_score | backed_up,19,27.703125,0.1300,-0.1300,-0.2339,-0.1081,0.0584,more_dropback_heavy_than_baseline,less_efficient_than_baseline
1992,leading_two_plus_scores | backed_up,13,27.991379,-0.1419,0.1419,0.1250,0.2370,-0.0681,more_run_heavy_than_baseline,more_efficient_than_baseline
1803,two_minute_half | backed_up,8,31.092742,-0.2563,0.2563,0.2027,0.2706,-0.0387,more_run_heavy_than_baseline,more_efficient_than_baseline


## 6. Sample Size Guardrails
These checks help keep us honest about situations that may be noisy because of smaller samples.

### How to read this section

These tables show where the project’s signals are backed by strong sample and where they are more fragile.

#### Why this matters for CPU-controlled coaching decisions

A team might look very distinctive in a tiny sample, but that does not always mean the signal is strong enough to tune around.

For design purposes:
- large, stable samples are better candidates for tuning
- smaller samples can still be interesting, but they should carry less weight

#### Practical takeaway

The best CPU tuning candidates are situations where:
- the team differs from baseline
- the behavior appears effective
- and the sample is strong enough to trust

In [106]:
ranked_situation_scores["team_sample_quality"].value_counts()

team_sample_quality
thin         702
strong       623
good         576
very_thin    557
Name: count, dtype: int64

In [107]:
ranked_situation_scores.loc[
    ranked_situation_scores["team_sample_quality"].isin(["thin", "very_thin"])
].sort_values(["team_sample_quality", "team_play_count", "team"]).head(30)

,rank,team,situation_order,situation_name,field_zone_order,field_zone,situation_field_zone_context,team_play_count,team_sample_quality,coach_dna_score_adjusted,coach_dna_score_raw,tendency_signal_score,efficiency_signal_score,explosiveness_signal_score,stability_signal_score,sample_reliability_score,dropback_rate_delta,rush_rate_delta,shotgun_rate_delta,no_huddle_rate_delta,avg_epa_delta,success_rate_delta,explosive_play_rate_delta,tendency_profile_label,formation_profile_label,tempo_profile_label,efficiency_profile_label
822,823,ARI,20,tied_early_down,4,red_zone,tied_early_down | red_zone,20,thin,47.545898,63.394531,79.687500,47.265625,62.500000,35.937500,55.0,0.2882,-0.2882,0.1250,-0.0782,-0.3284,-0.0809,-0.0005,more_dropback_heavy_than_baseline,more_shotgun_than_baseline,slower_than_baseline,less_efficient_than_baseline
614,615,ATL,9,two_minute_half,4,red_zone,two_minute_half | red_zone,20,thin,51.164062,68.218750,57.421875,78.515625,84.375000,73.437500,55.0,-0.1863,0.1863,0.1121,-0.0044,0.3719,0.0987,0.0552,more_run_heavy_than_baseline,more_shotgun_than_baseline,close_to_baseline_tempo,more_efficient_than_baseline
1599,1600,ATL,20,tied_early_down,4,red_zone,tied_early_down | red_zone,20,thin,34.245117,45.660156,59.375000,12.890625,62.500000,35.937500,55.0,-0.0618,0.0618,0.3250,0.0718,-0.3702,-0.0809,-0.0005,more_run_heavy_than_baseline,more_shotgun_than_baseline,faster_than_baseline,less_efficient_than_baseline
1964,1965,ATL,17,leading_two_plus_scores,3,fringe,leading_two_plus_scores | fringe,20,thin,28.448589,37.931452,35.887097,20.967742,58.064516,50.806452,55.0,0.0136,-0.0136,0.0600,-0.0677,-0.4620,-0.0345,0.0043,close_to_baseline_run_pass_split,more_shotgun_than_baseline,slower_than_baseline,less_efficient_than_baseline
918,919,BAL,20,tied_early_down,4,red_zone,tied_early_down | red_zone,20,thin,45.773438,61.031250,84.765625,26.171875,61.458333,43.750000,55.0,-0.1618,0.1618,-0.3250,-0.0782,0.0160,-0.0809,-0.0005,more_run_heavy_than_baseline,less_shotgun_than_baseline,slower_than_baseline,close_to_baseline_efficiency
1580,1581,BAL,17,leading_two_plus_scores,1,backed_up,leading_two_plus_scores | backed_up,20,thin,34.519397,46.025862,27.155172,60.775862,75.862069,44.827586,55.0,0.0004,-0.0004,-0.1462,-0.0278,0.0829,0.0216,0.1550,close_to_baseline_run_pass_split,less_shotgun_than_baseline,slower_than_baseline,more_efficient_than_baseline
1490,1491,CAR,17,leading_two_plus_scores,3,fringe,leading_two_plus_scores | fringe,20,thin,35.903226,47.870968,72.983871,26.209677,5.376344,49.193548,55.0,-0.0864,0.0864,-0.2400,-0.0677,-0.1584,0.0155,-0.1457,more_run_heavy_than_baseline,less_shotgun_than_baseline,slower_than_baseline,less_efficient_than_baseline
825,826,CHI,22,trailing_early_down,1,backed_up,trailing_early_down | backed_up,20,thin,47.487305,63.316406,79.687500,45.703125,61.458333,40.625000,55.0,-0.2432,0.2432,-0.2565,-0.0195,-0.1001,0.0789,0.0122,more_run_heavy_than_baseline,less_shotgun_than_baseline,close_to_baseline_tempo,less_efficient_than_baseline
2185,2186,CHI,10,two_minute_game,4,red_zone,two_minute_game | red_zone,20,thin,24.123992,32.165323,11.290323,41.532258,64.516129,42.741935,55.0,0.0000,0.0000,-0.1156,0.0471,-0.3258,-0.0283,0.0067,close_to_baseline_run_pass_split,less_shotgun_than_baseline,faster_than_baseline,less_efficient_than_baseline
1442,1443,CLE,20,tied_early_down,4,red_zone,tied_early_down | red_zone,20,thin,36.720703,48.960938,50.000000,50.781250,28.125000,67.968750,55.0,-0.1118,0.1118,-0.1250,-0.0282,0.1445,0.0191,-0.0505,more_run_heavy_than_baseline,less_shotgun_than_baseline,slower_than_baseline,more_efficient_than_baseline


In [108]:
small_sample_summary = (
    ranked_situation_scores
    .groupby(["situation_name", "team_sample_quality"], as_index=False)
    .size()
    .sort_values(["situation_name", "team_sample_quality"])
)

small_sample_summary.head(50)

,situation_name,team_sample_quality,size
0,all_offense,good,31
1,all_offense,strong,97
2,early_down,good,35
3,early_down,strong,90
4,early_down,thin,3
5,fourth_down,thin,3
6,fourth_down,very_thin,101
7,goal_line,thin,13
8,goal_line,very_thin,19
9,goal_to_go,good,6


In [109]:
ranked_situation_scores.groupby(
    ["situation_name", "field_zone", "team_sample_quality"],
    as_index=False
).size().sort_values(["situation_name", "field_zone", "team_sample_quality"])

,situation_name,field_zone,team_sample_quality,size
0,all_offense,backed_up,good,31
1,all_offense,backed_up,strong,1
2,all_offense,fringe,strong,32
3,all_offense,own_territory,strong,32
4,all_offense,red_zone,strong,32
...,...,...,...,...
162,two_minute_half,fringe,thin,24
163,two_minute_half,own_territory,good,15
164,two_minute_half,own_territory,thin,17
165,two_minute_half,red_zone,thin,18


## 7. Quick Finding Builder
These tables help turn the output into portfolio-ready statements.

In [110]:
top_overall = ranked_team_summary.head(10)[
    [
        "rank",
        "team",
        "overall_coach_dna_score",
        "score_tier",
        "top_signal_context",
        "top_signal_score",
    ]
]
top_overall

,rank,team,overall_coach_dna_score,score_tier,top_signal_context,top_signal_score
0,1,LA,56.2940,strong_signal,neutral_early_down | fringe,80.6250
1,2,BUF,53.1461,solid_signal,leading_early_down | own_territory,83.5156
2,3,WAS,51.9778,solid_signal,neutral_early_down | own_territory,83.3984
3,4,CIN,50.8784,solid_signal,trailing_one_score | fringe,71.6406
4,5,BAL,49.1322,solid_signal,neutral_early_down | fringe,77.4219
5,6,NE,49.0881,solid_signal,leading_one_score | fringe,66.8164
6,7,CHI,48.8438,solid_signal,trailing_one_score | own_territory,68.3203
7,8,SF,48.3072,solid_signal,neutral_early_down | own_territory,75.7227
8,9,SEA,47.6358,solid_signal,third_down | own_territory,70.9242
9,10,DET,46.0869,moderate_signal,leading_early_down | own_territory,68.3594


In [111]:
best_situations = (
    ranked_situation_scores
    .sort_values("coach_dna_score_adjusted", ascending=False)
    .head(15)[[
        "rank",
        "team",
        "situation_field_zone_context",
        "team_play_count",
        "coach_dna_score_adjusted",
        "tendency_profile_label",
        "efficiency_profile_label",
    ]]
)

best_situations

,rank,team,situation_field_zone_context,team_play_count,coach_dna_score_adjusted,tendency_profile_label,efficiency_profile_label
0,1,BUF,leading | own_territory,154,87.890625,more_run_heavy_than_baseline,more_efficient_than_baseline
1,2,WAS,early_down | own_territory,344,84.960938,more_run_heavy_than_baseline,more_efficient_than_baseline
2,3,BUF,leading_early_down | own_territory,127,83.515625,more_run_heavy_than_baseline,more_efficient_than_baseline
3,4,WAS,neutral_early_down | own_territory,181,83.398438,more_run_heavy_than_baseline,more_efficient_than_baseline
4,5,BAL,one_score | fringe,211,83.125000,more_run_heavy_than_baseline,more_efficient_than_baseline
5,6,BUF,all_offense | own_territory,441,83.007812,more_run_heavy_than_baseline,more_efficient_than_baseline
6,7,BAL,all_offense | fringe,282,82.617188,more_run_heavy_than_baseline,more_efficient_than_baseline
7,8,WAS,all_offense | own_territory,421,82.460938,more_run_heavy_than_baseline,more_efficient_than_baseline
8,9,BUF,early_down | own_territory,356,82.285156,more_run_heavy_than_baseline,more_efficient_than_baseline
10,11,LA,trailing | fringe,128,82.070312,more_run_heavy_than_baseline,more_efficient_than_baseline


In [112]:
most_run_heavy = (
    ranked_situation_scores
    .sort_values("rush_rate_delta", ascending=False)
    .head(15)[[
        "team",
        "situation_field_zone_context",
        "team_play_count",
        "rush_rate_delta",
        "avg_epa_delta",
        "success_rate_delta",
    ]]
)

most_run_heavy

,team,situation_field_zone_context,team_play_count,rush_rate_delta,avg_epa_delta,success_rate_delta
1400,NE,fourth_down | backed_up,1,0.7692,2.7850,0.5128
2013,NE,two_minute_game | backed_up,3,0.6988,-0.1523,-0.4710
1610,PIT,two_minute_game | backed_up,1,0.6988,0.1048,0.5290
1771,ATL,two_minute_game | backed_up,4,0.6988,0.0414,-0.2210
1987,BAL,two_minute_game | red_zone,3,0.6000,-0.6987,-0.3783
1577,ATL,two_minute_game | red_zone,6,0.6000,0.0596,0.1217
1839,LA,two_minute_game | red_zone,2,0.6000,-2.2458,0.1217
1185,JAX,fourth_down | own_territory,3,0.5938,2.4944,0.4723
1285,IND,tied | backed_up,1,0.5494,1.0547,0.6157
1302,IND,tied_early_down | backed_up,1,0.4855,1.0427,0.6074


In [113]:
most_dropback_heavy = (
    ranked_situation_scores
    .sort_values("dropback_rate_delta", ascending=False)
    .head(15)[[
        "team",
        "situation_field_zone_context",
        "team_play_count",
        "dropback_rate_delta",
        "avg_epa_delta",
        "success_rate_delta",
    ]]
)

most_dropback_heavy

,team,situation_field_zone_context,team_play_count,dropback_rate_delta,avg_epa_delta,success_rate_delta
2140,CIN,short_yardage | backed_up,1,0.5799,-1.1710,-0.5833
2230,LV,short_yardage | backed_up,5,0.5799,-1.2232,-0.3833
1441,NYG,short_yardage | backed_up,1,0.5799,1.9150,0.4167
1616,SF,short_yardage | backed_up,4,0.5799,0.7407,0.1667
1458,HOU,short_yardage | backed_up,1,0.5799,1.4479,0.4167
2051,CHI,short_yardage | backed_up,2,0.5799,-0.4579,-0.5833
2073,NYJ,short_yardage | backed_up,1,0.5799,-0.9194,-0.5833
1655,CAR,short_yardage | backed_up,1,0.5799,1.0051,0.4167
2160,TEN,fourth_down | own_territory,9,0.4062,-0.9461,-0.3055
2222,PIT,two_minute_game | red_zone,4,0.4000,-1.0946,-0.1283


## 8. Draft Findings

Use this section to capture the upgraded model’s strongest findings for the README, GitHub post, and interview narrative.

### Draft findings
- Field position materially improved the model. Some of the strongest average separation across teams appears in contexts like `all_offense | own_territory`, `all_offense | fringe`, and `early_down | own_territory`.
- The upgraded model shows that coaching DNA is not just about game situation. It is also about where the offense is operating on the field.
- Buffalo ranks near the top overall and shows one of its clearest signals in own-territory contexts, especially when leading or operating on early downs.
- Splitting scoring territory into `red_zone`, `goal_to_go`, and `goal_line` makes the model more football-realistic by separating compressed-field environments that were previously blended together.
- The strongest tuning opportunities are the contexts with both strong separation and strong sample. Those are the places where CPU-controlled coaching logic can become more authentic without overfitting to noise.
- The model now gives a studio a clearer answer to this question: not just how should a team behave, but how should that team behave here, in this exact game and field-position context.